In [40]:
import pandas as pd
import altair as alt

df = pd.read_csv("lung_cancer.csv")

pivot = df.groupby(["exercise_hours_per_week", "bmi"]).agg(
    risk=("lung_cancer_risk", "mean"),
    n=("lung_cancer_risk", "count")
).reset_index()

dense = pivot[pivot["n"] >= 3]

heatmap = alt.Chart(dense).mark_rect().encode(
    x=alt.X("bmi:O", title="BMI", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("exercise_hours_per_week:O", title="Exercise hours/week", sort="descending"),
    color=alt.Color(
        "risk:Q",
        scale=alt.Scale(scheme="blues", domain=[0, 0.5], reverse=False),
        legend=alt.Legend(title="Lung cancer risk")
    ),
    tooltip=[
        alt.Tooltip("exercise_hours_per_week:O", title="Exercise hrs/week"),
        alt.Tooltip("bmi:O", title="BMI"),
        alt.Tooltip("risk:Q", title="Risk", format=".1%"),
        alt.Tooltip("n:Q", title="Patients")
    ]
)

chart = (heatmap).properties(
    width=650,
    height=300,
    title="Lung Cancer Risk by Exercise and BMI"
)

chart.show()

alt.Chart(...)

In [49]:
pivot = df.groupby(["exercise_hours_per_week", "cigarettes_per_day"]).agg(
    risk=("lung_cancer_risk", "mean"),
    n=("lung_cancer_risk", "count")
).reset_index()

dense = pivot[pivot["n"] >= 3]

heatmap = alt.Chart(dense).mark_rect().encode(
    x=alt.X("cigarettes_per_day:O", title="Cigarettes/day", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("exercise_hours_per_week:O", title="Exercise hours/week", sort="descending"),
    color=alt.Color(
        "risk:Q",
        scale=alt.Scale(scheme="blues", domain=[0, 1], reverse=False),
        legend=alt.Legend(title="Lung cancer risk")
    ),
    tooltip=[
        alt.Tooltip("exercise_hours_per_week:O", title="Exercise hrs/week"),
        alt.Tooltip("cigarettes_per_day:O", title="Cigarettes/day"),
        alt.Tooltip("risk:Q", title="Risk", format=".1%"),
        alt.Tooltip("n:Q", title="Patients")
    ]
)

chart = (heatmap).properties(
    width=1000,
    height=300,
    title="Lung Cancer Risk by Exercise and Smoking Habits"
)

chart.show()

alt.Chart(...)

In [48]:
pivot = df.groupby(["bmi", "cigarettes_per_day"]).agg(
    risk=("lung_cancer_risk", "mean"),
    n=("lung_cancer_risk", "count")
).reset_index()

dense = pivot[pivot["n"] >= 3]

heatmap = alt.Chart(dense).mark_rect().encode(
    x=alt.X("cigarettes_per_day:O", title="Cigarettes/day", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("bmi:O", title="BMI", sort="descending"),
    color=alt.Color(
        "risk:Q",
        scale=alt.Scale(scheme="blues", domain=[0, 1], reverse=False),
        legend=alt.Legend(title="Lung cancer risk")
    ),
    tooltip=[
        alt.Tooltip("bmi:O", title="BMI"),
        alt.Tooltip("cigarettes_per_day:O", title="Cigarettes/day"),
        alt.Tooltip("risk:Q", title="Risk", format=".1%"),
        alt.Tooltip("n:Q", title="Patients")
    ]
)

chart = (heatmap).properties(
    width=600,
    height=300,
    title="Lung Cancer Risk by BMI and Smoking Habits"
)

chart.show()

alt.Chart(...)

In [69]:
pivot = df.groupby(["alcohol_units_per_week", "cigarettes_per_day"]).agg(
    risk=("lung_cancer_risk", "mean"),
    n=("lung_cancer_risk", "count")
).reset_index()

dense = pivot[pivot["n"] >= 3]

heatmap = alt.Chart(dense).mark_rect().encode(
    x=alt.X("cigarettes_per_day:O", title="Cigarettes/day", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("alcohol_units_per_week:O", title="Alcohol units/week", sort="descending"),
    color=alt.Color(
        "risk:Q",
        scale=alt.Scale(scheme="blues", domain=[0, 1], reverse=False),
        legend=alt.Legend(title="Lung cancer risk")
    ),
    tooltip=[
        alt.Tooltip("alcohol_units_per_week:O", title="Alcohol units/week"),
        alt.Tooltip("cigarettes_per_day:O", title="Cigarettes/day"),
        alt.Tooltip("risk:Q", title="Risk", format=".1%"),
        alt.Tooltip("n:Q", title="Patients")
    ]
)

chart = (heatmap).properties(
    width=600,
    height=300,
    title="Lung Cancer Risk by Drinking and Smoking Habits"
)

chart.show()

alt.Chart(...)

In [68]:
categories = {
    "BMI": "bmi",
    "Exercise hours/week": "exercise_hours_per_week",
    "Diet quality": "diet_quality",
    "Alcohol units/week": "alcohol_units_per_week"
}

vals = list(categories.values())
risk_col = "lung_cancer_risk"
df["risk_label"] = df[risk_col].map({0: "Not at risk", 1: "At risk"})
df["risk_label_num"] = df[risk_col]  # keeps original 0/1 integers
df = df.reset_index(drop=True)
df["patient_id"] = df.index

yparam = alt.param(
    name="yvar",
    value=categories["Exercise hours/week"],
    bind=alt.binding_select(options=vals, labels=list(categories.keys()), name="Y-axis: ")
)

fold_vars = vals + ["cigarettes_per_day"]

base = alt.Chart(df).add_params(yparam).transform_fold(
    fold_vars, as_=["variable", "value"]
)

data = (
    base
    .transform_filter(
        (alt.datum.variable == "cigarettes_per_day") | (alt.datum.variable == yparam)
    )
    .transform_calculate(
        axis="if(datum.variable === 'cigarettes_per_day', 'x', 'y')"
    )
    .transform_aggregate(
        val="max(value)",
        groupby=["patient_id", "axis", "risk_label_num"]
    )
    .transform_pivot(
        "axis", value="val", groupby=["patient_id", "risk_label_num"]
    )
)

heatmap = (
    data
    .mark_rect()
    .encode(
        alt.X("x:Q", bin=alt.Bin(step=1), title="Cigarettes per day"),
        alt.Y("y:Q", bin=alt.Bin(step=1)),
        alt.Color(
            "mean(risk_label_num):Q",
            title="Lung Cancer Risk",
            scale=alt.Scale(scheme="reds", domain=[0, 1])
        ),
        tooltip=[
            {"field": "x", "type": "quantitative", "title": "Cigarettes/day"},
            {"field": "y", "type": "quantitative", "title": "Y-axis value"},
            {"aggregate": "mean", "field": "risk_label_num", "type": "quantitative", "title": "Avg risk", "format": ".2f"},
            {"aggregate": "count", "type": "quantitative", "title": "Patients in bin"}
        ])
    
.properties(width=1000, height=500, title="Cigarettes/day vs Other Lifestyle Choices")
)

heatmap

alt.Chart(...)